In [6]:
import os
from glob import glob
import pandas as pd

In [ ]:
# 특정 경로 파일
# os 이용
os.listdir(
    "./"
)

In [19]:
# glob 이용
# 장점 : 파일의 이름을 하나의 리스트로 생성,
#         특정 확장자 파일만 추출 가능
json_list = glob("./*.json")

In [26]:
# json_list를 이용하여 하나의 데이터프레임으로 결합

total_df = pd.DataFrame()

for file in json_list:
    # print(file)
    df = pd.read_json(file)
    # total_df, df를 단순 행결합을 하여 total_df에 저장
    total_df = pd.concat([total_df, df], axis=-0)
    # print(df)
    # break
total_df.reset_index(drop=True, inplace=True)

In [29]:
total_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1423 entries, 0 to 1422
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            1423 non-null   int64  
 1   RawText          1423 non-null   object 
 2   Source           1423 non-null   object 
 3   Domain           1423 non-null   object 
 4   MainCategory     1423 non-null   object 
 5   ProductName      1423 non-null   object 
 6   Syllable         1423 non-null   int64  
 7   Word             1423 non-null   int64  
 8   GeneralPolarity  1418 non-null   float64
 9   Aspects          1423 non-null   object 
dtypes: float64(1), int64(3), object(6)
memory usage: 111.3+ KB


In [31]:
# Aspects 의 데이터를 하나로 합치고 새로운 데이터프레임을 생성
aspect_df = pd.DataFrame(total_df['Aspects'].sum())

In [32]:
aspect_df['SentimentPolarity'].value_counts()

SentimentPolarity
1     9664
-1    1005
0      293
Name: count, dtype: int64

In [38]:
aspect_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10962 entries, 0 to 10961
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Aspect             10962 non-null  object
 1   SentimentText      10962 non-null  object
 2   SentimentWord      10962 non-null  object
 3   SentimentPolarity  10962 non-null  object
dtypes: object(4)
memory usage: 342.7+ KB


In [39]:
# 데이터셋에서 문자열의 좌우의 공백 제거
# 모든 컬럼이 Object 타입이므로 strip 바로 사용 가능
aspect_df = aspect_df.map(lambda x: x.strip())

In [42]:
aspect_df.isin(['']).sum()

Aspect               0
SentimentText        0
SentimentWord        0
SentimentPolarity    0
dtype: int64

In [43]:
aspect_df['SentimentText'].value_counts()

SentimentText
가볍고                                           38
따뜻하고                                          25
저렴한 가격에                                       15
가격도 저렴하고                                      12
시원하고                                          11
                                              ..
비싼 밍크코트의 품질을 기대하지는 마시길 바랍니다.                   1
디자인이 너무너무 예쁩니다.                                1
색상도 고급스러워서 마음에 들구요.                            1
바느질도 꼼꼼하게 잘 되어 있어요.                            1
딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서     1
Name: count, Length: 10467, dtype: int64

In [ ]:
before_cnt = len(aspect_df)

aspect_df.drop_duplicates('SentimentText', inplace=True)

after_cnt = len(aspect_df)

print(f"중복 제거 전 데이터 건수 : {before_cnt} \n중복 제거 후 데이터 건수 : {after_cnt}")

중복 제거 전 데이터 건수 : 10962 
중복 제거 후 데이터 건수 : 10467


In [45]:
print(before_cnt - after_cnt)

495


In [46]:
# 1,0, -1 값의 비율 확인
aspect_df['SentimentPolarity'].value_counts()


SentimentPolarity
1     9183
-1     994
0      290
Name: count, dtype: int64

In [49]:
# 인덱스 초기화
aspect_df.reset_index(drop=True, inplace=True)

In [52]:
# 토큰화 -> 벡터화
from konlpy.tag import Komoran
from sklearn.feature_extraction.text import TfidfVectorizer

komoran = Komoran()
allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'SL']

def komoran_tokenize(text):
    tokens = []
    for word, pos in komoran.pos(text):
        if(pos in allow_pos) & (len(word) >= 2) :
            tokens.append(word)
        return tokens

vectorizer = TfidfVectorizer(
    tokenizer= komoran_tokenize,
    ngram_range=(1,2),
    min_df= 3,
    max_df= 0.8,
    max_features= 30000
)



In [51]:
# 모델 생성
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier

In [53]:
svc = LinearSVC(
    random_state=42,
    class_weight= 'balanced'
)

multi_model = MultiOutputClassifier(svc)

pipe = Pipeline(
    [
        ('vector', vectorizer),
        ('model', multi_model)
    ]
)

In [ ]:
from sklearn.model_selection import iter

In [65]:
# 데이터 폴드화 (계층화 폴드)
from sklearn.model_selection import StratifiedKFold, KFold

skfold = KFold(n_splits=3, shuffle= True, random_state=42)

In [66]:
from sklearn.preprocessing import LabelEncoder


In [67]:
le = LabelEncoder()

In [68]:
aspect_df['Aspect'] = le.fit_transform(aspect_df['Aspect'])
aspect_df['SentimentPolarity'] = aspect_df['SentimentPolarity'].astype('int')

In [69]:
# 독립 변수, 종속 변수 생성
X = aspect_df['SentimentText'].values
Y  = aspect_df[['Aspect', 'SentimentPolarity']].values

In [70]:
from sklearn.model_selection import GridSearchCV

In [71]:
params = {
    'model__estimator__C' : [1.0, 2.0]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid= params,
    cv= skfold,
    scoring='accuracy'
)

In [72]:
grid.fit(X,Y)

c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py:953: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 942, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_scorer.py", line 308, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^

,estimator,Pipeline(step..._state=42)))])
,param_grid,"{'model__estimator__C': [1.0, 2.0]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [73]:
grid.best_score_

np.float64(nan)

1. total_df 에서 rawText 컬럼의 데이터들을 Kkma를 이용하여 문장별로 나눠줌
2. 이용하여 grid의best_estimator_ 에서 예측을 실행
3. 실행된 결과 값을 이용하여 데이터프레임( RawText, Aspect_pred, Pola_pred )으로 생성
4. rawText, Aspect_pred 값을 이용하여 그룹화 -> 그룹화 연산에는 평균

In [74]:
best_model = grid.best_estimator_

In [78]:
from konlpy.tag import Kkma

In [79]:
kkma = Kkma()

In [98]:
kkma.sentences(total_df.loc[1, 'RawText'])

['드디어 겨울이 찾아왔네요.',
 '이제부터 슬슬 겨울 패딩 장만하셔야지요?',
 '패딩 소개해 드릴게요.',
 '작년부터 눈여겨 보던',
 'OO 입니다.',
 '저는 블랙 90 사이즈 구매했는데 딱 잘 맞네요.',
 '작년에 비해 저렴하게 구매해서 아주 만족해요.',
 '신제품은 경량 패딩으로 되 서 3in1 기능이더라',
 '구요.',
 '경량 패딩 구매 따로 하셔야 하는 분들에 겐 신제품 매장에서 확인하시라 고 권하고 싶네요.',
 '저는 경량이 있는 관계로 이 제품 구매했는데 슬림 핏이라 날씬해 보여요.',
 '하지만 안에 두껍게 입 기엔 좀 불편하더라구요.',
 '간편하게 니트 티셔츠에 입기에는 딱입니다.',
 '아직은 초겨울이라 OO를 쭉 입고 있어요.',
 '후기 읽으며 많이 고민했고 도움을 받아서 저도 후기 남겨 봅니다.',
 '누군가에 겐 도움이 되셨기를 바라는 마음에서요. OO 패딩 좋은 건 다들 아실 테고.. 롱이면서 슬림하니 감추고 싶은 부분 감추고 그러면서도 날씬하게 보이는 패딩으로 이번 겨울 멋쟁이가 되어 보시 길 요..']

In [100]:
text = []

for i in range(len(total_df)):
    rawtext = kkma.sentences(total_df.loc[i, 'RawText'])
    text.append(rawtext)

In [ ]:
text

In [ ]:
pd.DataFrame(text)